# GIAC Cancer Subtype Classification
Workflow: Mount Drive → Pull GitHub → Install → Train → Evaluate

## 1. Mount Kaggle

In [ ]:
import os

# Đường dẫn dataset trên Kaggle (nhớ đổi 'giac-dataset' thành tên bạn đã đặt ở Bước 1)
DATA_DIR  = '/kaggle/input/datasets/maivanquan/datn-2025-2/data_final'
GRAPH_DIR = '/kaggle/input/datasets/maivanquan/datn-2025-2/Heterogeneous_Graph'

print('── data_final ──')
if os.path.exists(DATA_DIR):
    for f in sorted(os.listdir(DATA_DIR)):
        size = os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6
        print(f'  {f:45s} {size:.1f} MB')
else:
    print(f"Không tìm thấy {DATA_DIR}. Hãy check lại tên dataset!")

print('\n── Heterogeneous_Graph ──')
if os.path.exists(GRAPH_DIR):
    for f in sorted(os.listdir(GRAPH_DIR)):
        fpath = os.path.join(GRAPH_DIR, f)
        if os.path.isfile(fpath):
            size = os.path.getsize(fpath) / 1e6
            print(f'  {f:45s} {size:.1f} MB')
        else:
            print(f'  {f}/')
            for ff in sorted(os.listdir(fpath)):
                size = os.path.getsize(os.path.join(fpath, ff)) / 1e6
                print(f'    {ff:43s} {size:.1f} MB')

## 2. Clone / Pull repo từ GitHub

In [ ]:
REPO_URL = 'https://github.com/maivanquan-00/giac_project_kaggle.git'

import os
# Đổi /content sang /kaggle/working
if os.path.exists('/kaggle/working/giac_project_kaggle'):
    %cd /kaggle/working/giac_project_kaggle
    !git pull
else:
    %cd /kaggle/working
    !git clone {REPO_URL}
    %cd /kaggle/working/giac_project_kaggle
!ls

## 3. Install dependencies

In [ ]:
import torch
TORCH = torch.__version__.split('+')[0]
CUDA  = 'cu121' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {TORCH} | CUDA: {CUDA}')

!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-{TORCH}+{CUDA}.html -q
!pip install torch-geometric entmax gseapy pyyaml tqdm scikit-learn -q
print('Done!')

## 4. Kiểm tra GPU

In [ ]:
import torch
print(f'CUDA : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU  : {torch.cuda.get_device_name(0)}')
    print(f'VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

## 5. Train

In [ ]:
# Đã sửa lỗi /content/giac_project thành /kaggle/working/giac_project_kaggle
%cd /kaggle/working/giac_project_kaggle
!python train.py --config configs/config.yaml

## 6. Evaluate + top-K gene per patient

In [ ]:
!python evaluate.py \
    --config configs/config.yaml \
    --checkpoint checkpoints/best_model.pt \
    --split test \
    --top_k 50

## 7. Xem patient report

In [ ]:
import pandas as pd
df = pd.read_csv('patient_features.csv')
print(f'Total: {len(df)} | Accuracy: {df["correct"].mean():.4f}')
display(df.head(10))

## 8. Lưu kết quả về Drive

In [ ]:
import os

# Chỉ cần đảm bảo các file kết quả đang nằm trong /kaggle/working hoặc thư mục con của nó
print("Các file kết quả sẽ được Kaggle tự động lưu. Danh sách file hiện tại:")
!ls -lh /kaggle/working/giac_project_kaggle/patient_features.csv
!ls -lh /kaggle/working/giac_project_kaggle/checkpoints/best_model.pt